# Quantitative Trading Strategies — 2025/2026 Edition

This notebook explains all **20 modern quantitative strategies** implemented in `strategy.py`, organized across **8 strategy types**.

Each section covers:
- What the strategy does and **why it works**
- Key parameters and entry/exit logic
- Live signal demo using real market data
- Typical Sharpe ratio, risk, and notable firms

---

| # | Strategy | Type | Complexity | Sharpe |
|---|----------|------|------------|--------|
| 1 | Time-Series Momentum | Momentum | Low | 0.8–1.5 |
| 2 | Cross-Sectional Momentum | Momentum | Medium | 0.8–1.3 |
| 3 | ML-Enhanced Trend | Momentum | High | 0.8–1.5 |
| 4 | Short-Term Reversal | Momentum | Medium | 0.5–1.2 |
| 5 | Pairs Trading | Stat Arb | Medium | 1.0–2.0 |
| 6 | ETF Arbitrage | Stat Arb | Very High | 2.0–5.0 |
| 7 | PCA Basket Arb | Stat Arb | Very High | 1.5–3.0 |
| 8 | Crypto Basis Arb | Stat Arb | Medium | 1.5–3.0 |
| 9 | Gradient Boosting Alpha | ML | High | 1.0–2.0 |
| 10 | Deep RL Agent | ML | Very High | 0.8–2.0 |
| 11 | LLM Earnings Alpha | ML | High | 1.0–2.5 |
| 12 | Alt Data Satellite | ML | Very High | 1.2–2.5 |
| 13 | VRP Harvesting | Volatility | High | 0.8–1.5 |
| 14 | Dispersion Trading | Volatility | Very High | 0.8–1.5 |
| 15 | Gamma Scalping | Volatility | High | 0.5–1.5 |
| 16 | Multi-Factor L/S | Factor | Medium | 0.8–1.5 |
| 17 | Factor Momentum | Factor | Low | 0.6–1.2 |
| 18 | Social/News Sentiment | Sentiment | High | 0.8–1.5 |
| 19 | Electronic Market Making | Market Making | Extreme | 3.0–10.0 |
| 20 | Earnings Surprise Drift | Event-Driven | Medium | 1.0–2.0 |

## Setup

In [ ]:
import sys, os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('.'))

from strategy import (
    list_strategies, get_strategy, STRATEGY_REGISTRY,
    TimeSeriesMomentum, CrossSectionalMomentum, MLEnhancedTrend, ShortTermReversal,
    PairsTrading, ETFArbitrage, PCABasketArb, CryptoBasisArb,
    GradientBoostingAlpha, DeepRLAgent, LLMEarningsAlpha, AltDataSatellite,
    VolatilityRiskPremium, DispersionTrading, GammaScalping,
    MultiFactorLongShort, FactorMomentum,
    SocialNewsSentiment, ElectronicMarketMaking, EarningsSurprise
)
from datafetch import fetch

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('All imports OK')
print(f'\n{len(STRATEGY_REGISTRY)} strategies loaded:')
display(list_strategies())

## Helper — Fetch Data + Plot

In [ ]:
def demo_signal(strategy, symbol, days=300, title_prefix=''):
    """Fetch data, run strategy, plot price + signal."""
    df = fetch(symbol, days=days)
    if df is None or df.empty:
        print(f'  No data for {symbol}')
        return None, None
    
    result = strategy.generate_signal(df, symbol)
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                             gridspec_kw={'height_ratios': [3, 1]})
    
    axes[0].plot(df.index, df['close'], color='steelblue', lw=1.5, label='Price')
    color = {'BUY': 'green', 'SELL': 'red', 'NEUTRAL': 'gray', 'HOLD': 'orange'}.get(result.signal, 'gray')
    axes[0].axhline(df['close'].iloc[-1], color=color, lw=2, ls='--', alpha=0.7,
                    label=f'Signal: {result.signal} (strength={result.strength:.2f})')
    axes[0].set_title(f'{title_prefix}{strategy.__class__.__name__} — {symbol}', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Price')
    axes[0].legend(loc='upper left')
    
    axes[1].bar(df.index, df['volume'], color='steelblue', alpha=0.5, label='Volume')
    axes[1].set_ylabel('Volume')
    
    plt.tight_layout()
    plt.show()
    
    print(f'  Signal: {result}')
    print(f'  Indicators: {result.indicators}')
    if result.notes:
        print(f'  Notes: {result.notes}')
    return df, result

def load_meta(strategy_id, class_name):
    """Load and display JSON metadata for a strategy."""
    fname = f'strategies/{strategy_id:02d}_{class_name.lower()}.json'
    if not os.path.exists(fname):
        # Try partial match
        import glob
        matches = glob.glob(f'strategies/{strategy_id:02d}_*.json')
        if matches:
            fname = matches[0]
    if os.path.exists(fname):
        with open(fname) as f:
            return json.load(f)
    return {}

def show_meta(meta):
    """Pretty-print key metadata fields."""
    if not meta:
        return
    fields = ['name', 'type', 'subtype', 'assets', 'timeframe', 'sharpe_ratio',
              'signal_buy', 'signal_sell', 'implementation_complexity', 'notable_firms']
    data = {k: meta[k] for k in fields if k in meta}
    display(pd.DataFrame.from_dict(data, orient='index', columns=['Value']))

print('Helper functions ready.')

---
# TYPE 1 — Momentum / Trend Following

**Core idea:** Markets trend. Assets that went up continue going up (in the medium term). This is one of the oldest and most robust phenomena in finance, documented across 200+ years and all asset classes.

**Why it works:** Investor underreaction to news → slow diffusion of information → trends persist for weeks to months. Herding behavior amplifies and extends trends.

**Risk:** "Momentum crashes" — sharp reversals after market panics or central bank interventions.

## Strategy 1 — Time-Series Momentum (TSMOM)

The simplest and most powerful trend strategy: look at an asset's **own** 12-month return. If positive → long, if negative → short. Each position is sized by inverse volatility so all markets contribute equal risk.

**Used by:** Man AHL, Winton Capital, Aspect Capital (CTAs managing $10B–$50B each)

In [ ]:
meta = load_meta(1, 'timeseriesmomen')
show_meta(meta)
strat = TimeSeriesMomentum()
print(f'\nStrategy description: {strat.describe()}\n')
df, result = demo_signal(strat, 'SPY', title_prefix='[1] ')

## Strategy 2 — Cross-Sectional Equity Momentum (12-1 Factor)

Instead of trading each asset on its own, **rank** a universe of 500+ stocks by their 12-month return (skipping last month to avoid reversal). Long the top 20%, short the bottom 20%. This is the classic "WML" (Winners Minus Losers) factor.

**Key insight:** Stocks that outperformed last year tend to continue outperforming over the next 1–3 months.

In [ ]:
meta = load_meta(2, 'crosssectionalmom')
show_meta(meta)
strat = CrossSectionalMomentum()
print(f'\nDescription: {strat.describe()}\n')

# Compare two stocks: strong momentum vs weak momentum
symbols = ['NVDA', 'INTC']
for sym in symbols:
    df = fetch(sym, days=300)
    if df is not None:
        r = strat.generate_signal(df, sym)
        print(r)

## Strategy 3 — ML-Enhanced Trend Following

Traditional trend (EWMA crossover) **augmented with a machine learning quality filter**. The ML layer predicts whether a trend signal is likely to follow through or reverse, using features like cross-asset correlations, VIX term structure, and COT positioning data.

**Innovation:** Instead of just knowing the direction of the trend, the ML model scores the *quality* of the trend.

In [ ]:
meta = load_meta(3, 'mlenhancedtrend')
show_meta(meta)
strat = MLEnhancedTrend()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'QQQ', title_prefix='[3] ')

## Strategy 4 — Short-Term Reversal

The **opposite** of momentum at very short horizons (1–5 days). A stock that dropped 5% in the last week has a statistical tendency to bounce. This is driven by bid-ask bounce, forced selling, and short-term liquidity dynamics — not fundamentals.

**Caution:** Extremely high turnover (~300% monthly). Transaction costs are the primary performance driver.

In [ ]:
meta = load_meta(4, 'shorttermreversal')
show_meta(meta)
strat = ShortTermReversal()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'TSLA', title_prefix='[4] ')

---
# TYPE 2 — Statistical Arbitrage / Mean Reversion

**Core idea:** Some relationships between assets are stable over time (cointegration). When prices temporarily deviate from their equilibrium, they tend to revert. Trade the deviation.

**Why it works:** Temporary supply/demand imbalances, liquidity events, index rebalancing flows. The structural relationship (same business, same industry) is fundamentally anchored.

**Risk:** Relationships can permanently break down (M&A, regulatory change, sector disruption).

## Strategy 5 — Classic Pairs Trading

Find two cointegrated stocks. When their **log-price spread** diverges beyond 2 standard deviations, buy the cheap one and short the expensive one. Wait for reversion.

**Classic pairs:** Coca-Cola / PepsiCo, ExxonMobil / Chevron, Goldman Sachs / Morgan Stanley.

In [ ]:
meta = load_meta(5, 'pairstrading')
show_meta(meta)
strat = PairsTrading()
print(f'\nDescription: {strat.describe()}\n')

# Build a synthetic spread: KO vs PEP
ko  = fetch('KO',  days=300)
pep = fetch('PEP', days=300)

if ko is not None and pep is not None:
    # Align on common dates
    spread_df = pd.DataFrame({
        'close_a': ko['close'],
        'close_b': pep['close']
    }).dropna()
    result = strat.generate_signal(spread_df, 'KO/PEP')
    print(result)
    
    # Plot the spread
    import numpy as np
    log_spread = np.log(spread_df['close_a']) - np.log(spread_df['close_b'])
    mu  = log_spread.rolling(252).mean()
    std = log_spread.rolling(252).std()
    z   = (log_spread - mu) / std
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(spread_df.index, spread_df['close_a'], label='KO', color='red')
    axes[0].plot(spread_df.index, spread_df['close_b'], label='PEP', color='blue')
    axes[0].set_title('[5] Pairs Trading — KO / PEP', fontsize=13, fontweight='bold')
    axes[0].legend()
    
    axes[1].plot(z.index, z, color='purple', lw=1.5, label='Z-score')
    axes[1].axhline(2.0,  color='red',   ls='--', alpha=0.7, label='+2σ entry (SELL A)')
    axes[1].axhline(-2.0, color='green', ls='--', alpha=0.7, label='-2σ entry (BUY A)')
    axes[1].axhline(0,    color='gray',  ls='-',  alpha=0.3)
    axes[1].set_ylabel('Z-Score')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## Strategy 6 — ETF Arbitrage

ETFs must trade close to their Net Asset Value (NAV). When they deviate, **authorized participants** can create/redeem ETF shares to profit and simultaneously close the gap. This strategy monitors and trades those deviations.

**Who dominates:** Jane Street, Citadel Securities — this is a core business generating billions annually.

In [ ]:
meta = load_meta(6, 'etfarbitrage')
show_meta(meta)
strat = ETFArbitrage()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'SPY', title_prefix='[6] ')

## Strategy 7 — Multi-Leg Statistical Arbitrage (PCA-Based)

Instead of pairs, use **Principal Component Analysis** to decompose an entire market's returns into common factors (sector risk, market risk, etc.). Trade the residuals — what's left after removing all common factors — when they deviate.

**Scale:** Citadel, Millennium, D.E. Shaw run thousands of simultaneous positions using this approach.

In [ ]:
meta = load_meta(7, 'pcabasketarb')
show_meta(meta)
strat = PCABasketArb()
print(f'\nDescription: {strat.describe()}\n')

# Single-asset fallback: AAPL
df, result = demo_signal(strat, 'AAPL', title_prefix='[7] ')

## Strategy 8 — Crypto Basis / Funding Rate Arbitrage

Unique to crypto perpetual futures. **Long spot BTC + short BTC perpetual** = delta neutral. In bull markets, the funding rate paid by longs to shorts is positive (e.g. 0.03% per 8 hours = ~13% annualized yield). Collect this yield with zero directional exposure.

**2025 performance:** Documented 31%+ cumulative returns in 2025 with very low drawdowns.

In [ ]:
meta = load_meta(8, 'cryptobasisarb')
show_meta(meta)
strat = CryptoBasisArb()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'BTCUSDT', title_prefix='[8] ')

---
# TYPE 3 — Machine Learning / AI Strategies

**Core idea:** Use ML models (gradient boosting, deep learning, LLMs) to find alpha in large, complex datasets that traditional rule-based approaches cannot process.

**Why it works:** Markets are not fully efficient. ML can find subtle non-linear patterns across hundreds of features simultaneously.

**Risk:** Overfitting, distribution shift, black-box opacity. ML strategies require constant monitoring and retraining.

## Strategy 9 — Gradient Boosting Ensemble Alpha (GBEA)

Train XGBoost / LightGBM on **150+ features** (technical, fundamental, alternative data) to predict 5-day forward returns. The ensemble approach prevents any single tree from overfitting. Long top-20% scored stocks, short bottom-20%.

**Why GBDT?** Handles missing values, non-linear relationships, and feature interactions naturally. SHAP values provide interpretability.

In [ ]:
meta = load_meta(9, 'gradientboostingalpha')
show_meta(meta)
strat = GradientBoostingAlpha()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'NVDA', title_prefix='[9] ')

# Show all features computed
if df is not None:
    close = df['close'].astype(float)
    feats = strat._build_features(close)
    print('\nAll computed features (last 5 rows):')
    display(feats.tail())

## Strategy 10 — Deep Reinforcement Learning (DRL) Agent

An RL agent learns to trade by **trial and error** in a simulated market environment. State = current market features. Actions = {buy, sell, hold}. Reward = risk-adjusted return. The agent discovers optimal trading rules without being programmed with explicit rules.

**Key challenge:** Financial markets have very limited training data relative to other RL domains (unlike Go, which has billions of self-play games).

In [ ]:
meta = load_meta(10, 'deeprlagent')
show_meta(meta)
strat = DeepRLAgent()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'AAPL', title_prefix='[10] ')

# Visualize state vector
if df is not None:
    close = df['close'].astype(float)
    state = strat._state(close)
    print('\nCurrent state vector fed to RL agent:')
    for k, v in state.items():
        print(f'  {k:<20}: {v:.4f}')

## Strategy 11 — LLM-Driven Earnings and Filings Alpha

Large language models (Claude, GPT-4o) **read earnings transcripts** and detect subtle changes in management tone, new risk factors, and forward guidance. They generate a sentiment score that predicts post-earnings drift before the market fully prices in the information.

**Edge:** LLMs process the entire 10-K in seconds. Humans reading 100 pages miss subtle signals. The alpha decays within 1–3 trading days.

In [ ]:
meta = load_meta(11, 'llmearningsalpha')
show_meta(meta)
strat = LLMEarningsAlpha()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'MSFT', title_prefix='[11] ')

## Strategy 12 — Alternative Data: Satellite + Credit Card Signal

**Satellite imagery** of parking lots at Walmart/Target tracks customer foot traffic in real-time — days before official earnings. **Credit card panels** from Affinity/Second Measure show exact consumer spending by company at near-real-time frequency.

**Cost:** These datasets cost $500K–$5M/year. Only accessible to well-funded hedge funds.

In [ ]:
meta = load_meta(12, 'altdatasatellite')
show_meta(meta)
strat = AltDataSatellite()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'AMZN', title_prefix='[12] ')

---
# TYPE 4 — Volatility Strategies

**Core idea:** Trade volatility itself, not price direction. Options markets systematically misprice volatility in predictable ways.

**Volatility Risk Premium:** Implied volatility (options prices) consistently exceeds realized volatility by 2–4 vol points. Selling options is like being an insurance company.

**Warning:** These strategies have **negative skewness** — small steady gains, but rare catastrophic losses (e.g. VIX spike of Feb 2018 wiped out multiple funds in one day).

## Strategy 13 — Volatility Risk Premium (VRP) Harvesting

Sell options (straddles, variance swaps, short VIX futures) when implied volatility is significantly higher than what's actually being realized. Collect the premium difference.

**Why it exists:** Investors pay a premium for crash protection → options are systematically overpriced → selling them has positive expected value.

In [ ]:
meta = load_meta(13, 'vrpharvesting')
show_meta(meta)
strat = VolatilityRiskPremium()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'SPY', title_prefix='[13] ')

# Plot: compare realized vol vs implied vol (VIX proxy)
if df is not None:
    close = df['close'].astype(float)
    rv_21 = close.pct_change().rolling(21).std() * np.sqrt(252) * 100
    iv_proxy = rv_21 * 1.15  # IV ≈ 115% of RV historically
    vrp = iv_proxy - rv_21
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(close.index, rv_21, label='Realized Vol (21d)', color='steelblue')
    axes[0].plot(close.index, iv_proxy, label='Implied Vol (proxy)', color='orange', ls='--')
    axes[0].set_title('[13] VRP Harvesting — SPY Volatility Surface', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Annualized Vol %')
    axes[0].legend()
    
    axes[1].fill_between(vrp.index, vrp, 0, where=(vrp > 0), alpha=0.6, color='green', label='VRP > 0 (sell vol)')
    axes[1].fill_between(vrp.index, vrp, 0, where=(vrp < 0), alpha=0.6, color='red',   label='VRP < 0 (buy vol)')
    axes[1].axhline(2.0,  color='green', ls='--', alpha=0.5, label='+2 vol pts threshold')
    axes[1].axhline(-2.0, color='red',   ls='--', alpha=0.5, label='-2 vol pts threshold')
    axes[1].set_ylabel('VRP (vol pts)')
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## Strategy 14 — Dispersion Trading

**Sell index volatility, buy individual stock volatilities.** Index implied vol is systematically overpriced relative to a basket of its constituent options. The correlation risk premium makes this persist.

**Intuition:** The index option buyer also gets crash protection from all stocks moving together — they pay extra for this correlation insurance.

In [ ]:
meta = load_meta(14, 'dispersiontrading')
show_meta(meta)
strat = DispersionTrading()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'QQQ', title_prefix='[14] ')

## Strategy 15 — Gamma Scalping (Long Gamma / Delta-Neutral)

**Buy options and delta-hedge continuously.** Each hedge locks in a small profit proportional to how much the stock moved squared (Gamma × ΔS²). Profit when the stock moves a lot (realized vol > implied vol paid).

**Opposing strategy to VRP:** Where VRP harvesting sells vol, gamma scalping buys it. One profits in calm markets; the other profits in volatile markets.

In [ ]:
meta = load_meta(15, 'gammascalping')
show_meta(meta)
strat = GammaScalping()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'NVDA', title_prefix='[15] ')

---
# TYPE 5 — Factor / Smart Beta Strategies

**Core idea:** Returns are driven by systematic risk factors. Tilt portfolios toward factors that have earned a premium historically: value, momentum, quality, low volatility, small size.

**Academic foundation:** Fama-French factor models. Today extended to 5-factor and beyond.

**2025 status:** Factor strategies face crowding as $1T+ in AUM now explicitly tracks factors. Alpha has compressed but factors persist.

## Strategy 16 — Multi-Factor Long-Short Equity (Quality + Value + Momentum)

Blend three uncorrelated factors: **Momentum** (past winners), **Value** (cheap stocks), **Quality** (profitable, stable companies). Each factor is orthogonal → the blend has higher Sharpe than any individual factor.

**AQR's insight (2013):** Momentum and value are negatively correlated. Combining them dramatically smooths returns.

In [ ]:
meta = load_meta(16, 'multifactorlongshort')
show_meta(meta)
strat = MultiFactorLongShort()
print(f'\nDescription: {strat.describe()}\n')

# Score multiple stocks
universe = ['AAPL', 'MSFT', 'NVDA', 'META', 'AMZN', 'INTC', 'IBM', 'T', 'KO', 'XOM']
scores = []
for sym in universe:
    df = fetch(sym, days=300)
    if df is not None:
        r = strat.generate_signal(df, sym)
        scores.append({'symbol': sym, 'signal': r.signal, 'strength': r.strength,
                       **{k: round(v, 3) for k, v in r.indicators.items()}})

scores_df = pd.DataFrame(scores).sort_values('strength', ascending=False)
print('\nMulti-Factor scores across universe:')
display(scores_df)

## Strategy 17 — Factor Momentum (Momentum of Momentum)

Apply the momentum concept to **factors themselves**. If the Value factor has been outperforming for the past 3 months, increase your value allocation. Simple but effective — factor returns show 3–12 month persistence.

**Simple implementation:** Buy factor ETFs when their 3-month return is positive; reduce exposure when negative.

In [ ]:
meta = load_meta(17, 'factormomentum')
show_meta(meta)
strat = FactorMomentum()
print(f'\nDescription: {strat.describe()}\n')

# Factor ETFs as proxies
factor_etfs = {'Momentum': 'MTUM', 'Value': 'VTV', 'Quality': 'QUAL',
               'Low Vol': 'USMV', 'Small Cap': 'IWM'}
print('Factor ETF momentum signals:')
for factor_name, sym in factor_etfs.items():
    df = fetch(sym, days=200)
    if df is not None:
        r = strat.generate_signal(df, sym)
        print(f'  {factor_name:<12} ({sym}): {r.signal:<8} | {r.indicators}')

---
# TYPE 6 — Sentiment / NLP

**Core idea:** Market prices reflect not just fundamentals but also sentiment and narrative. NLP models can extract this information from text data faster than humans.

**Data sources:** Twitter/X, Reddit, news wires (Bloomberg, Reuters), SeekingAlpha, earnings transcripts.

**2025 status:** FinBERT is now commoditized. Edge has shifted to LLM-based models that can handle nuanced context.

## Strategy 18 — Real-Time Social + News Sentiment Alpha

NLP models continuously score news articles and social media for each stock. The **decay-weighted aggregate score** drives buy/sell signals. Signal has a 1–3 day half-life — price usually fully absorbs sentiment within a week.

**Risk:** Social media manipulation. The 2021 GameStop/AMC episode showed coordinated Reddit buying can temporarily overwhelm institutional signals.

In [ ]:
meta = load_meta(18, 'socialnewssentiment')
show_meta(meta)
strat = SocialNewsSentiment()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'TSLA', title_prefix='[18] ')
print('\n(Note: Attach a real NLP feed to replace the volume proxy for production use)')

---
# TYPE 7 — Market Making / HFT

**Core idea:** Be the market. Post bid and ask quotes simultaneously. Earn the bid-ask spread on every transaction. This is not speculation — it's providing liquidity in exchange for a fee.

**Scale:** Jane Street reported **$10.1 billion** in net trading revenue in Q2 2025 alone — primarily from ETF market making.

**Barrier to entry:** Requires co-location, FPGA hardware, $1M+/year infrastructure. Not accessible to most traders.

## Strategy 19 — Electronic Market Making

Post bid/ask quotes around fair value. Cancel and requote in microseconds as price moves. The **Avellaneda-Stoikov model** optimally adjusts quotes based on inventory level and volatility.

**Key equation:** Each side of the spread earns ~half the spread on each fill. A 5bps spread on $100M/day = $2.5M/day before costs.

In [ ]:
meta = load_meta(19, 'electronicmarketmaking')
show_meta(meta)
strat = ElectronicMarketMaking()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'SPY', title_prefix='[19] ')

# Simulate simple spread economics
print('\n--- Simple Market Making P&L Simulation ---')
spread_bps = 5
daily_volume_m = 100   # $100M daily flow
fill_rate = 0.5        # fill both sides 50% of the time
adverse_pct = 0.20     # 20% of fills are adverse selection
gross_per_day = daily_volume_m * 1e6 * spread_bps / 10000 * fill_rate
adverse_loss  = daily_volume_m * 1e6 * spread_bps / 10000 * fill_rate * adverse_pct
print(f'  Gross spread income/day : ${gross_per_day:>12,.0f}')
print(f'  Adverse selection cost  : ${adverse_loss:>12,.0f}')
print(f'  Net income/day          : ${gross_per_day - adverse_loss:>12,.0f}')
print(f'  Annualized (250d)       : ${(gross_per_day - adverse_loss) * 250:>12,.0f}')

---
# TYPE 8 — Event-Driven

**Core idea:** Predictable corporate events create predictable price patterns. Earnings announcements, M&A, index additions/removals, spin-offs.

**Post-Earnings Announcement Drift (PEAD)** is one of the **oldest documented anomalies** in finance (Ball & Brown 1968) and remarkably still persists in 2025, especially in small and mid-cap stocks.

## Strategy 20 — Earnings Surprise + Post-Earnings Drift

When a stock **beats earnings expectations** AND the stock price gaps up on the day, there's a statistical tendency to continue drifting upward for 3–10 trading days. The market underreacts to earnings surprises.

**Why it persists:** Sell-side analysts revise estimates slowly. Fund managers take time to build positions. Information diffuses gradually.

In [ ]:
meta = load_meta(20, 'earningssurprise')
show_meta(meta)
strat = EarningsSurprise()
print(f'\nDescription: {strat.describe()}\n')
df, result = demo_signal(strat, 'META', title_prefix='[20] ')

---
# Full Daily Scan — All Strategies × All Symbols

In [ ]:
from datafetch import WATCHLIST, fetch

# Select a subset of liquid, easy-to-fetch strategies for the scan
scan_strategies = {
    'TimeSeriesMomentum':   TimeSeriesMomentum(),
    'MLEnhancedTrend':      MLEnhancedTrend(),
    'ShortTermReversal':    ShortTermReversal(),
    'GradientBoosting':     GradientBoostingAlpha(),
    'MultiFactorLS':        MultiFactorLongShort(),
    'EarningsSurprise':     EarningsSurprise(),
    'VRPHarvesting':        VolatilityRiskPremium(),
    'SocialSentiment':      SocialNewsSentiment(),
}

symbols = WATCHLIST['us_stocks'] + ['BTCUSDT', 'ETHUSDT']

print(f'Scanning {len(symbols)} symbols × {len(scan_strategies)} strategies...\n')

results = []
for sym in symbols:
    df = fetch(sym)
    if df is None or df.empty:
        continue
    row = {'symbol': sym}
    for strat_name, strat in scan_strategies.items():
        r = strat.generate_signal(df, sym)
        row[strat_name] = r.signal
    # Count buy/sell signals
    signals = [v for k, v in row.items() if k != 'symbol']
    row['buy_count']  = signals.count('BUY')
    row['sell_count'] = signals.count('SELL')
    row['price']      = round(float(df['close'].iloc[-1]), 2)
    results.append(row)

scan_df = pd.DataFrame(results)
scan_df = scan_df.sort_values('buy_count', ascending=False)
print('\nFull strategy scan results:')
display(scan_df)

In [ ]:
# Heatmap of signals
strat_cols = list(scan_strategies.keys())
heatmap_data = scan_df[['symbol'] + strat_cols].set_index('symbol')
heatmap_num  = heatmap_data.replace({'BUY': 1, 'SELL': -1, 'NEUTRAL': 0, 'HOLD': 0})
heatmap_num  = heatmap_num.fillna(0).astype(float)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(heatmap_num.values.T, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(heatmap_num.index)))
ax.set_yticks(range(len(strat_cols)))
ax.set_xticklabels(heatmap_num.index, rotation=30, ha='right')
ax.set_yticklabels([c.replace('_', ' ') for c in strat_cols])
plt.colorbar(im, ax=ax, label='Signal (-1=SELL, 0=NEUTRAL, +1=BUY)')
ax.set_title('Strategy Signal Heatmap — All Symbols × All Strategies', fontsize=13, fontweight='bold')

# Add text annotations
for i in range(len(heatmap_num.index)):
    for j in range(len(strat_cols)):
        val = heatmap_data.iloc[i, j]
        ax.text(i, j, str(val)[:1], ha='center', va='center', fontsize=8, color='black')

plt.tight_layout()
plt.show()

---
# Strategy Type Summary

In [ ]:
summary = [
    {'Type': 'Momentum',       'Count': 4, 'Typical Sharpe': '0.8–1.5', 'Crisis Alpha': 'Partial', 'Complexity': 'Low–High',     'Key Risk': 'Momentum crashes'},
    {'Type': 'Stat Arb',       'Count': 4, 'Typical Sharpe': '1.0–3.0', 'Crisis Alpha': 'No',      'Complexity': 'Medium–Very High', 'Key Risk': 'Cointegration breakdown / crowding'},
    {'Type': 'Machine Learning','Count': 4, 'Typical Sharpe': '0.8–2.5', 'Crisis Alpha': 'No',      'Complexity': 'High–Very High',  'Key Risk': 'Overfitting / distribution shift'},
    {'Type': 'Volatility',     'Count': 3, 'Typical Sharpe': '0.5–1.5', 'Crisis Alpha': 'Partial', 'Complexity': 'High–Very High',  'Key Risk': 'Vega blowup / vol spikes'},
    {'Type': 'Factor',         'Count': 2, 'Typical Sharpe': '0.6–1.5', 'Crisis Alpha': 'No',      'Complexity': 'Low–Medium',   'Key Risk': 'Factor crowding / regime shifts'},
    {'Type': 'Sentiment',      'Count': 1, 'Typical Sharpe': '0.8–1.5', 'Crisis Alpha': 'No',      'Complexity': 'High',         'Key Risk': 'Social manipulation / signal decay'},
    {'Type': 'Market Making',  'Count': 1, 'Typical Sharpe': '3.0–10',  'Crisis Alpha': 'No',      'Complexity': 'Extreme',      'Key Risk': 'Adverse selection / infrastructure'},
    {'Type': 'Event-Driven',   'Count': 1, 'Typical Sharpe': '1.0–2.0', 'Crisis Alpha': 'No',      'Complexity': 'Medium',       'Key Risk': 'Earnings reversals / hard-to-borrow'},
]

summary_df = pd.DataFrame(summary)
display(summary_df.set_index('Type'))

# Pie chart of strategy types
fig, ax = plt.subplots(figsize=(8, 6))
counts = summary_df.set_index('Type')['Count']
ax.pie(counts, labels=counts.index, autopct='%1.0f%%', startangle=90,
       colors=plt.cm.Set3.colors[:len(counts)])
ax.set_title('Distribution of Strategy Types (20 total)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()